# 🚀 Phase 2: Reference-Free Anchoring

This notebook isolates the first generated panel as the primary visual anchor and extracts identity embedding tokens (facial topology, wardrobe features) for downstream panels.

## 🔧 0. Universal Environment Setup
Run this cell first to configure Colab or local Jupyter environments.

In [ ]:
# ============================================================
# Universal Colab/Local Setup — run this first in every notebook
# ============================================================
import os, sys

try:
    from google.colab import files  # type: ignore
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    import subprocess
    _repo = "/content/Indie-Comic"
    if not os.path.exists(_repo):
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/Cyberpunk-San/Indie-Comic.git", _repo], check=True)
    exec(open(f"{_repo}/indie_comic_pipeline/colab_setup.py").read())
else:
    _candidates = [
        os.path.join(os.getcwd(), "colab_setup.py"),
        os.path.join(os.getcwd(), "indie_comic_pipeline", "colab_setup.py"),
    ]
    _found = next((p for p in _candidates if os.path.exists(p)), None)
    if _found:
        exec(open(_found).read())
    else:
        print("colab_setup.py not found — run from repo root")


In [ ]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Detected Google Colab. Setting up environment...")
    subprocess.run([sys.executable, "-c", "import urllib.request; exec(urllib.request.urlopen('https://raw.githubusercontent.com/Cyberpunk-San/Indie-Comic/main/indie_comic_pipeline/colab_setup.py').read())"], check=False)
else:
    print("💻 Detected Local Jupyter. Setting up path...")
    if os.path.exists("colab_setup.py"):
        exec(open("colab_setup.py").read())
    else:
        print("⚠️ colab_setup.py not found. Please ensure you are running from the correct directory.")

## ⚓ 1. Isolate Visual Anchor & Extract Identity Tokens

In [ ]:
import os
from PIL import Image
from core.memory import StorySectionMemory
from core.anchoring import ReferenceeFreeAnchor

memory = StorySectionMemory()
memory.register_character("Akira")

# Create mock panel 1 image
img = Image.new("RGB", (512, 512), (120, 140, 160))
anchor_dir = "outputs/anchors"
os.makedirs(anchor_dir, exist_ok=True)
mock_path = os.path.join(anchor_dir, "anchor_panel_1.png")
img.save(mock_path)

anchor_system = ReferenceeFreeAnchor(device="cpu")
tokens = anchor_system.establish_anchor(img, panel_id=1, character_name="Akira", memory=memory)

print("Anchor established. Extracted features:")
print("Aesthetic score:", tokens.get("aesthetic_score"))
print("Mean brightness:", tokens.get("mean_brightness"))